In [1]:
import sys
# !{sys.executable} -m pip install --upgrade pip
# !{sys.executable} -m pip install --upgrade python-dotenv
# !{sys.executable} -m pip install --upgrade edgartools
# !{sys.executable} -m pip install --upgrade chromadb
# !{sys.executable} -m pip install --upgrade langchain-huggingface
# !{sys.executable} -m pip install --upgrade langchain-chroma
# !{sys.executable} -m pip install --upgrade langchain-text-splitters
# !{sys.executable} -m pip install --upgrade langchain-groq

In [2]:
# Path to the chroma db
CHROMA_DB = './db/sec_rag'
# Name of the collection
COLLECTION_NAME = 'form-10k'

## Loads environment variables

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import chromadb

def count_collection():
    # Initialize the ChromaDB client (e.g., PersistentClient for local storage)
    client = chromadb.PersistentClient(path=CHROMA_DB)
    # Attempt to get the collection
    collection = client.get_collection(name=COLLECTION_NAME)
    print(collection.count())

In [5]:
# Just to ensure that we don't keep on adding documents with each run; we should have 6 documents
count_collection()

6


## Income Statement

In [6]:
from edgar import Company

def get_income_statement(ticker:str):
    company = Company(ticker)
    facts = company.get_facts()
    return facts.income_statement()

In [7]:
# Ticker symbol of the company we are interested
ticker = 'GOOG'
# Convert the income statement to LLM aware context
llm_context = get_income_statement(ticker=ticker).to_llm_context()

## Split into documents

In [8]:
from langchain_text_splitters import RecursiveJsonSplitter
from langchain_core.documents import Document
import json

json_splitter = RecursiveJsonSplitter(max_chunk_size=1000, min_chunk_size=50)
split_chunks = json_splitter.split_json(llm_context)

# documents = json_splitter.create_documents(texts=llm_context)

# Keep on getting IndexError when creating documents; so, let's create documents manually
documents = []
for i, chunk in enumerate(split_chunks):
    # Page content must be string, assign an id
    doc = Document(page_content=json.dumps(chunk), id=i)
    documents.append(doc)

## Suppress warnings from torch

In [9]:
import warnings
warnings.filterwarnings('ignore', module='torch')

## Create embeddings and vector store

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma 
from chromadb.config import Settings

# Our embeddings model
embeddings_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Vector store; data saved locally
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_model,
    persist_directory=CHROMA_DB    
)
# Persist documents
vector_store.add_documents(documents)

['0', '1', '2', '3', '4', '5']

## Define the LLM

In [11]:
from langchain_groq import ChatGroq

# Ensure your GROQ_API_KEY is set in environment variables
llm = ChatGroq(temperature=0.5, model_name='llama-3.1-8b-instant')

## Build the RAG Chain

In [12]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template for the LLM
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful financial assistant. Answer the query below based on the provided context:
    context: {context}
    query: {input}
    """
)
# Create a document chain to stuff retrieved documents into the prompt
document_chain = create_stuff_documents_chain(llm, prompt)

# Create a retriever from the vector store
retriever = vector_store.as_retriever()

# Combine the retriever and document chain into a RAG chain
qa_chain = create_retrieval_chain(retriever, document_chain)

## Periods

In [13]:
query = 'Which years are covered?'
response = qa_chain.invoke({'input': query})
print(response['answer'])

Based on the provided context, the years covered are:

- FY 2024
- FY 2023
- FY 2022
- FY 2021

These years are mentioned in the "periods" field of the second document and are also referenced in the other documents.


## Total Revenue

In [14]:
query = 'What are the total revenues in millions for 2024 and 2023?'
response = qa_chain.invoke({'input': query})
print(response['answer'])

To find the total revenues in millions for 2024 and 2023, we need to extract the relevant information from the provided context.

From the first context, we have:

- total_revenue_fy_2024 = 350018000000.0
- total_revenue_fy_2023 = 307394000000.0

To convert these values to millions, we divide by 1,000,000 (1 million).

- total_revenue_fy_2024 (in millions) = 350018000000.0 / 1000000 = 350,018
- total_revenue_fy_2023 (in millions) = 307394000000.0 / 1000000 = 307,394

So, the total revenues in millions for 2024 and 2023 are:

- 2024: 350,018 million
- 2023: 307,394 million


## Basic EPS

In [15]:
query = 'What is the basic earnings per share for 2024?'
response = qa_chain.invoke({'input': query})
print(response['answer'])

Based on the provided context, the basic earnings per share for 2024 is $8.13.


## EPS status

In [16]:
query = 'Has the basic earnings per share over the period improved or deteriorated?'
response = qa_chain.invoke({'input': query})
print(response['answer'])

Based on the provided data, we can analyze the trend of basic earnings per share (EPS) over the period.

Here are the basic EPS values for each year:

- FY 2021: 5.69
- FY 2022: 4.59
- FY 2023: 5.84
- FY 2024: 8.13

We can see that the basic EPS values have fluctuated over the period. 

Initially, the EPS decreased from FY 2021 to FY 2022 (5.69 to 4.59, a decrease of 1.10). 

However, the EPS then increased from FY 2022 to FY 2023 (4.59 to 5.84, an increase of 1.25) and further increased from FY 2023 to FY 2024 (5.84 to 8.13, an increase of 2.29).

Therefore, the basic earnings per share over the period have improved, with a significant increase in FY 2024 compared to the previous years.


## Cost of Revenue

In [17]:
query = 'Is there a reduction to the cost of revenue over the years?'
response = qa_chain.invoke({'input': query})
print(response['answer'])

To determine if there's a reduction to the cost of revenue over the years, we can calculate the cost of revenue for each year and compare the values.

From the provided data, we can see that the cost of revenue for each year is as follows:

- FY 2021: $110,939,000,000
- FY 2022: $126,203,000,000
- FY 2023: $133,332,000,000
- FY 2024: $146,306,000,000

To check for a reduction, we can calculate the percentage change in cost of revenue from one year to the next.

- FY 2021 to FY 2022: ($126,203,000,000 - $110,939,000,000) / $110,939,000,000 = 13.7% increase
- FY 2022 to FY 2023: ($133,332,000,000 - $126,203,000,000) / $126,203,000,000 = 5.5% increase
- FY 2023 to FY 2024: ($146,306,000,000 - $133,332,000,000) / $133,332,000,000 = 9.8% increase

Based on these calculations, the cost of revenue has increased over the years, not decreased.
